
# assignment


In [72]:

#i = iterations (int)
#k = motif len (int)
#p = pseudocount (float)

#fastareader goes here. i: file path; o: "header", "sequence" variables which are strings of the header and sequence of the fasta
#we probably don't need the header for this assignment but I will keep it in the code


import os
os.chdir('C:/Users/liang/Downloads')

########################################################################
# File:program.py
#  executable: program.py
# Purpose:scaffold file for python executables
#   stderr: errors and status
#   stdout:
#          
# Author: David Bernick
# History:      dlb 08/20/2011 Created
#               
########################################################################

########################################################################
# CommandLine
########################################################################
class CommandLine() :
    '''
    Handle the command line, usage and help requests.

    CommandLine uses argparse, now standard in 2.7 and beyond. 
    it implements a standard command line argument parser with various argument options,
    a standard usage and help, and an error termination mechanism do-usage_and_die.

    attributes:
    all arguments received from the commandline using .add_argument will be
    avalable within the .args attribute of object instantiated from CommandLine.
    For example, if myCommandLine is an object of the class, and requiredbool was
    set as an option using add_argument, then myCommandLine.args.requiredbool will
    name that option.
 
    '''
    
    def __init__(self, inOpts=None) :
        '''
        CommandLine constructor.
        
        Implements a parser to interpret the command line argv string using argparse.
        '''
        
        import argparse
        self.parser = argparse.ArgumentParser(description = 'Program prolog - a brief description of what this thing does', 
                                             epilog = 'Program epilog - some other stuff you feel compelled to say', 
                                             add_help = True, #default is True 
                                             prefix_chars = '-', 
                                             usage = '%(prog)s [options] -option1[default] <input >output' 
                                             )
        #self.parser.add_argument('-o', '--option', action = 'store', help='foo help')
        self.parser.add_argument('-i', '--integer', type=int, action = 'store', help='number of iterations')
        self.parser.add_argument('-k', '--kmer', type=int, action = 'store', help='motif length')
        self.parser.add_argument('-p', '--pseudocount', type=float, action = 'store', help='pseudocount')
        self.parser.add_argument('-g', '--gibbs', action = 'store', nargs='?', required=False,const=True, default=False, help='Use Gibbs sampling to find the optimal consensus motif.')
        self.parser.add_argument('-m', '--list', action = 'store', nargs='?', required=False,const=True,default=False, help='Print the specific motif and the name of the contributing sequence for each of the participating sequences.')
        #self.parser.add_argument('-v', '--version', action='version', version='%(prog)s 0.1')  
        if inOpts is None :
            self.args = self.parser.parse_args()
        else :
            self.args = self.parser.parse_args(inOpts)
  
       

########################################################################

#FastaReader
#               
########################################################################

class FastAreader :
	
    def __init__ (self, fname=''):
        '''contructor: saves attribute fname '''
        self.fname = fname
            
    def doOpen (self):
        if self.fname is '':
            return sys.stdin
        else:
            return open(self.fname)
 
    def readFasta (self):
		
        header = ''
        sequence = ''
        
        with self.doOpen() as fileH:
			
            header = ''
            sequence = ''
 
            # skip to first fasta header
            line = fileH.readline()
            while not line.startswith('>') :
                line = fileH.readline()
            header = line[1:].rstrip()

            for line in fileH:
                if line.startswith ('>'):
                    yield header,sequence
                    header = line[1:].rstrip()
                    sequence = ''
                else :
                    sequence += ''.join(line.rstrip().split()).upper()
						
        yield header,sequence
        
########################################################################
# File: SequenceParser.py
#  executable: SequenceParser.py
# Purpose: grab headers only and sequences only from fastareader; then grab random motifs of size k for each sequence
#   stderr: errors and status
#   stdout:
#          
# Author: Cindy Liang
# History:      dlb 08/20/2011 Created
#               
########################################################################

class SequenceParser:
    def __init__ (self, generator):
        self.generator = generator
    
    def grabSequences(generator):
        sequenceList = []
        for seq_tuple in generator:        
            sequenceList.append(seq_tuple[1])
                
        return sequenceList
    
    def grabHeader(generator):
        headerList = []
        for header_tuple in generator:
            headerList.append(header_tuple[1])
        return headerList        
    
# calculate null model for a set of sequences
    
    def nullModel(sequence, k,p):
        for string in sequence:
            countA = p
            countC = p
            countG = p
            countT = p
            countA += string.count("A")
            countC += string.count("C")
            countG += string.count("G")
            countT += string.count("T")
            counts = [countA, countC, countG, countT]
            
            N = countA + countC + countG + countT
            
            QA = countA/(N+4*p)
            QC = countC/(N+4*p)
            QG = countG/(N+4*p)
            QT = countT/(N+4*p)
            counts[:] = [QA,QC,QG,QT]
        
        
                
        return counts
        counts = counts.reshape(nrows, ncols)
    
# grab random motif from each sequence for first profile 

    def randomMotif(sequence, k):
        
        minseqlen = len(min(sequence, key=len))
        
        randommotifs = []
        rng = numpy.random.default_rng(12345)
        rints = rng.integers(low=0, high=minseqlen-k)
        for a in sequence:
            randommotifs.append(a[rints:rints+k])
            numpy.array(randommotifs)
        return randommotifs
                       
# from list of random motifs, obtain counts
# first, turn motifs list ['aaaaaa', 'aaaaaa', 'aaaaa'] into an array so I can do column operations on it. 
#then, organize motifs by columns so I can count the number of nucleotides per column

    def motifByColumn(motifs, sequence, k):
                        
        motifarray = []
        motifcols = []
        motifcolstring = []
        #first grab columns from each motif in motif list
        
        for motif in motifs:
            motif = list(motif)
            motifarray.append(motif)   
        motifarray = numpy.char.array(motifarray)
          
        for i in range(0,k):
                    
            x = (motifarray[:,i])
                
            motifcols.append(x)
        
        for array in motifcols:
            array = array.tolist()
            motifcolstring.append(array)
        return motifcolstring
    
    def countMatrix(listostr,p):
        
        countdict = {"A": [] , "C": [] , "G": [] , "T": [] }
        for string in listostr:
            countdict["A"].append(string.count("A")+p)
            countdict["C"].append(string.count("C")+p)
            countdict["G"].append(string.count("G")+p)
            countdict["T"].append(string.count("T")+p)
                        
        return countdict    
    
    def findProfile(countdict,k):
        N = (countdict['A'][0] + countdict['G'][0] + countdict['C'][0] + countdict['T'][0])
        for i in range(0,k):
            probA = countdict["A"][i]/N
            countdict["A"][i] = probA            
            probC = countdict["C"][i]/N
            countdict["C"][i] = probC
            probG = countdict["G"][i]/N
            countdict["G"][i] = probG
            probT = countdict["T"][i]/N
            countdict["T"][i] = probT
        return countdict    
            
    #given a set of profiles, select best kmer motif using RE equation
    
    def findMotifs(countdict, seqlist, k):
        product = 1
        motifdict = {}
        newmotifmatrix = []
        
        seqstrings = []
        for alist in seqlist:
             astring = ''.join(item for item in alist)
             seqstrings.append(astring)
                
        for i in range (0,k):
            for string in seqstrings:
                product = 1
                product *= countdict[string[i]][i] 
                
            
            motifdict[string[i:i+k]] = product
            newmotifmatrix.append(max(motifdict))
        return newmotifmatrix              
    
    def findConsensus(countsmatrix,k):
        consensus = ""         
        for i in range(0,k):
            bestcount = -1
            bestbase = -1
                   
            
            for base in ["A","C","G","T"]:
                count = countsmatrix.get(base)[i]
            
                if count > bestcount:
                    bestcount = count
                    bestbase = base
            consensus += bestbase
        return consensus
    
    def outerLoop(i, p, k, oldmotif, newmotif, seqdata, nullprob):
        
#         test1 = FastAreader(r'C://Users//liang//Downloads//allFilesCrisprs.fa')
#         seq = test1.readFasta()
#         test2 = SequenceParser(seq)
            
#         sequencedata = SequenceParser.grabSequences(seq)
#         null = SequenceParser.nullModel(sequencedata, k, p)
        
        for iteration in range (0,i):
            while oldmotif != newmotif:
                oldmotif = newmotif
                
                #iterate through inner loop to generate a new, new motif
                
                motifcols2 = SequenceParser.motifByColumn(oldmotif, seqdata, k)
 
                counts2 = SequenceParser.countMatrix(motifcols2,p)
 
                profile2 = SequenceParser.findProfile(counts2,k)
    
                newmotif = SequenceParser.findMotifs(profile2, motifcols2, k)
        
         #find REs once oldmotif = newmotif
            for x in range (0,k):          
                bestRE = -1
                RE = 0
                RE += numpy.log2(profile["A"][x]/null[0]) + numpy.log2(profile["C"][x]/null[1]) + \
                numpy.log2(profile["G"][x]/null[2]) + numpy.log2(profile["T"][x]/null[3])
                score = RE
            
            
            if RE > bestRE:
                bestRE = RE
                bestConsensus = SequenceParser.findConsensus(counts,k)
        print(bestConsensus, "\t", bestRE)
               
##### main stuff ############

def main():
    import numpy
    import random
    
    
    i = 250
    p = 1
    k = 13         
    test1 = FastAreader(r'C://Users//liang//Downloads//allFilesCrisprs.fa')
    seq = test1.readFasta()
    test2 = SequenceParser(seq)
    
    sequencedata = SequenceParser.grabSequences(seq)
    #return sequencedata
    print("list of seqs is ",sequencedata)


    randmotifs = SequenceParser.randomMotif(sequencedata, k)
    print("list of random motifs is ",randmotifs)

    null = SequenceParser.nullModel(sequencedata, k, p)
    print("null Qs are ",null)

    motifcols = SequenceParser.motifByColumn(randmotifs, sequencedata, k)
    print("random motifs arranged by columns are ",motifcols)

    counts = SequenceParser.countMatrix(motifcols,p)
    print("initial count matrix is ",counts)
    # this is old profile

    profile = SequenceParser.findProfile(counts,k)
    print("prob matrix is", profile)

    motifdict = SequenceParser.findMotifs(profile, motifcols, k)
    print("new motifs are ",motifdict)
    
    newmotifcols = SequenceParser.motifByColumn(motifdict, sequencedata, k)
    print("new motifs arranged by columns are ",)

    newcounts = SequenceParser.countMatrix(newmotifcols, p)
    print("new counts are ",newcounts)
    
    newprofile = SequenceParser.findProfile(newcounts, k)
    print("new profile is ", newprofile)
    
    outerLoop = SequenceParser.outerLoop(i, p, k, randmotifs, motifdict, sequencedata, null)
    print(outerLoop)  
        
    
main()
    #CommandLine("-i=10000 -p=1 -k=13")

 
    


list of seqs is  ['AAAAAACAGACGAAAAACTTAAAAACCCACAAAAACAATCAAAACCAGCC', 'AAACCTACAAAAAACTTAAAAACCCACAAAAACCAACAAAACCAGCCCCA', 'CAAAAAGCAGACGAAAAACTTAAAAACCCACAAAAACCATCAAAACCAGC', 'TACACAAGAAAAACTTAAAAACCCACAAAAAACCATGACCAGGAACCCCA', 'CGAAACATTTATAAAAGAGCAAAAACCAAAATCCCAAATAAAAACCCACA', 'CGAAACATTTATAAAAGAGCAAAAACCAAAATCCCAAATAAAAACCCACA', 'CGAAACATTTATAAAAGAGCAAAAACTAAGATCCCAAATAAAAACCCACA', 'AAAAAACAGACGAAAAACTTAAAAACACACAAAAACCGCAAAAACCAGCC', 'AAAAAATCAAAGAAAAACTTAAAAACCCACAGGAACTATACACGCAAGCC', 'AAAAAATCAAAGAAAAACTTAAAAACCCACAGGAACTATACACGCAAGCC', 'AAAAGGAGAAAAACTTAAAAACCCCCAAAAACACAAATCCAGAGATCCCA', 'TCTACCTCGCGACAATCAGCAGAGAGCTAGACGAGATAAGAAAGGAAATA', 'CAAAAAATTTATAAAGTGGGAAAAACTAACATCCCGAATAAAAACCCACA', 'CGTATCCTTAATACTTCGACATCCTTGCTCTGTACCAGAAAGTAAACATC', 'AAAAAACCAAAGAAAAACTTAAAAACCCAAAAGAACTTTAAACGCAAGCC', 'CACCCAAGAAAAACTTAAAAACCCACCAAAACAAACCCCCAGAAACCCCA', 'CACCCAAGAAAAACTTAAAAACCCACCAAAACAAACCCCCAGAAACCCCA', 'AAAACTAAAGAAAAACTTAAAAACCCAAAAAAACCACCCACATAAGCGGT', 'AAAAGAAAA

<>:85: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:85: SyntaxWarning: "is" with a literal. Did you mean "=="?
<ipython-input-72-dcef2af4a4cc>:85: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if self.fname is '':


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (13,) + inhomogeneous part.


# this is fastareader.py


In [ ]:
import sys
class FastAreader :
	
    def __init__ (self, fname=''):
        '''contructor: saves attribute fname '''
        self.fname = fname
            
    def doOpen (self):
        if self.fname is '':
            return sys.stdin
        else:
            return open(self.fname)
 
    def readFasta (self):
		
        header = ''
        sequence = ''
        
        with self.doOpen() as fileH:
			
            header = ''
            sequence = ''
 
            # skip to first fasta header
            line = fileH.readline()
            while not line.startswith('>') :
                line = fileH.readline()
            header = line[1:].rstrip()

            for line in fileH:
                if line.startswith ('>'):
                    yield header,sequence
                    header = line[1:].rstrip()
                    sequence = ''
                else :
                    sequence += ''.join(line.rstrip().split()).upper()
						
        yield header,sequence

#then, count patterns in the sequence string

class patternCount():
    
    #class attribute -- stick things you want to stay constant per patternCount here
    #cant think of anything for now
    count == o
    
    #define init to set the initial state of the object
    
    def .__init__(self, sequence, motif)
    
        #create the sequence and motif instance attributes
        
        self.sequence = sequence
        self.motif = motif
    
    #add a count for each motif found
    
    for i in range(0, |sequence|-|motif|, 1):
        if sequence(i,|motif|) = motif
            count = count + 1
    
    return count     
        
#instantiate patternCounts by calling the class
#I will need to call patternCounts at least 2x -- one for null model and one for the actual data
patternCounts()

#instantiate patternCounts for null where randomsequence = random null seq ; k = int len of motif
nullprofile = patternCounts(randomsequence, k)

#instantiate patternCounts for expmt where fastafile = path to file ; k = int len of motif
queryprofile = patternCounts(fastafile, k)


# this is the code that lets you type command line things into stdin


In [ ]:
#!/usr/bin/env python3

########################################################################
# File:program.py
#  executable: program.py
# Purpose:scaffold file for python executables
#   stderr: errors and status
#   stdout:
#          
# Author: David Bernick
# History:      dlb 08/20/2011 Created
#               
########################################################################

########################################################################
# CommandLine
########################################################################
class CommandLine() :
    '''
    Handle the command line, usage and help requests.

    CommandLine uses argparse, now standard in 2.7 and beyond. 
    it implements a standard command line argument parser with various argument options,
    a standard usage and help, and an error termination mechanism do-usage_and_die.

    attributes:
    all arguments received from the commandline using .add_argument will be
    avalable within the .args attribute of object instantiated from CommandLine.
    For example, if myCommandLine is an object of the class, and requiredbool was
    set as an option using add_argument, then myCommandLine.args.requiredbool will
    name that option.
 
    '''
    
    def __init__(self, inOpts=None) :
        '''
        CommandLine constructor.
        
        Implements a parser to interpret the command line argv string using argparse.
        '''
        
        import argparse
        self.parser = argparse.ArgumentParser(description = 'Program prolog - a brief description of what this thing does', 
                                             epilog = 'Program epilog - some other stuff you feel compelled to say', 
                                             add_help = True, #default is True 
                                             prefix_chars = '-', 
                                             usage = '%(prog)s [options] -option1[default] <input >output' 
                                             )
        self.parser.add_argument('-o', '--option', action = 'store', help='foo help')
        self.parser.add_argument('-i', '--integer', type=int, choices= range(5, 10), action = 'store', help='help for a boolean option')
        self.parser.add_argument('-c', '--character', choices ='abcdef', action = 'store', help='help for a charcter option')
        self.parser.add_argument('-b', '--bool', action = 'store', nargs='?', const=True, default=False, help='boolean switch')
        self.parser.add_argument('-r', '--requiredBool', action = 'store', nargs='?', required=True,const=True, default=False, help='required boolean switch')
        self.parser.add_argument('-l', '--list', action = 'append', nargs='?', help='list help') #allows multiple list options
        self.parser.add_argument('-v', '--version', action='version', version='%(prog)s 0.1')  
        if inOpts is None :
            self.args = self.parser.parse_args()
        else :
            self.args = self.parser.parse_args(inOpts)
  

class Usage(Exception):
    '''
    Used to signal a Usage error, evoking a usage statement and eventual exit when raised.
    '''
    def __init__(self, msg):
        self.msg = msg 
########################################################################
# Main
# Here is the main program
# 
#
########################################################################
   

def main(myCommandLine=None):
    '''
    Implement the Usage exception handler that can be raised from anywhere in process.

    '''
    if myCommandLine is None:
        myCommandLine = CommandLine()  # read options from the command line
    else :
        myCommandLine = CommandLine(myCommandLine) # interpret the list passed from the caller of main as the commandline.

    try:
        
        print (myCommandLine.args) # print the parsed argument string .. as there is nothing better to do

        if myCommandLine.args.requiredBool:
            print ('requiredBool is', str(myCommandLine.args.requiredBool) )  ## this is just an example
        else :
            pass
        raise Usage ('testing')  # this is an example of how to raise a Usage exception and you can include some text that will get printed. Delete this is you dont need it

    except Usage as err:
       print (err.msg)

if __name__ == "__main__":
    main()
#   main(['-r'])  # this would make this program.py behave as written

# if you want to make use of a test commandLine, you could do it like this ( notice that it is a list of strings ):
#   main([ '--bool',
#          '--character=b',
#          '-i=10'])

